# Case Study: IMDB Movie Dataset Analysis

Analyze movie ratings, genres, budgets, and trends using synthetic movie data.
This notebook demonstrates EDA, visualization, and insight extraction.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

movies = pd.read_csv('Data/movies.csv')
reviews = pd.read_csv('Data/reviews.csv')
print('Movies shape:', movies.shape)
print('Reviews shape:', reviews.shape)


## 1. Initial Exploration


In [ ]:
movies.head()


In [ ]:
movies.info()
print('\n---')
movies.describe()


In [ ]:
reviews.head()


## 2. Genre Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=movies, x='genre', order=movies['genre'].value_counts().index)
plt.title('Movie Count by Genre')
plt.xticks(rotation=45)
plt.show()


## 3. Budget Trends Over Time


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=movies, x='year', y='budget_millions')
plt.title('Budget Distribution by Year')
plt.xticks(rotation=90)
plt.show()


In [ ]:
# Average budget by genre
movies.groupby('genre')['budget_millions'].agg(['mean', 'median', 'count']).round(1).sort_values('mean', ascending=False)


## 4. Rating Analysis


In [ ]:
merged = reviews.merge(movies[['movie_id', 'title', 'genre', 'year']], on='movie_id')

plt.figure(figsize=(10, 4))
sns.histplot(merged['rating'], bins=20, kde=True)
plt.title('Distribution of Movie Ratings')
plt.show()


In [ ]:
# Average rating by genre
merged.groupby('genre')['rating'].agg(['mean', 'std', 'count']).round(2).sort_values('mean', ascending=False)


## 5. Year-over-Year Trends


In [ ]:
yearly = merged.groupby('year').agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count'),
    avg_budget=('budget_millions', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(yearly['year'], yearly['avg_rating'])
axes[0].set_title('Avg Rating Over Time')
axes[1].plot(yearly['year'], yearly['num_ratings'])
axes[1].set_title('Number of Ratings Over Time')
axes[2].plot(yearly['year'], yearly['avg_budget'])
axes[2].set_title('Avg Budget Over Time')
plt.tight_layout()
plt.show()


## 6. Top Movies by Rating


In [ ]:
movie_stats = merged.groupby(['movie_id', 'title', 'genre']).agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

top_10 = movie_stats.sort_values('avg_rating', ascending=False).head(10)
top_10


In [ ]:
# Movies with highest number of ratings (most popular)
movie_stats.sort_values('num_ratings', ascending=False).head(10)


## 7. Correlation Analysis


In [ ]:
numeric_cols = movies.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(8, 6))
sns.heatmap(movies[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()


## Key Insights


In [ ]:
print('=== Key Insights ===')
print(f"Total movies analyzed: {len(movies)}")
print(f"Total reviews analyzed: {len(reviews)}")
genre_counts = movies['genre'].value_counts()
print(f"Most common genre: {genre_counts.index[0]} ({genre_counts.values[0]} movies)")
print(f"Average movie rating: {merged['rating'].mean():.2f}/10")
print(f"Highest avg rating genre: {merged.groupby('genre')['rating'].mean().idxmax()}")
